# WORCAP — pipeline final reproduzível

Este notebook apenas orquestra o código versionado. Ele **não envia submissões**. Execute as fases na ordem, examine os backtests e peça aprovação antes de fazer upload de qualquer CSV ao Kaggle.

In [ ]:
from pathlib import Path
import os, shutil, subprocess, sys, json

DATA = Path('/kaggle/input/previsao-climatica-de-precipitacao-sobre-a-america-do-sul')
REPO = Path('/kaggle/working/desafiodogoverno')
if not (REPO / 'scripts/worcap_pipeline.py').exists():
    extracted = list(Path('/kaggle/input').rglob('scripts/worcap_pipeline.py'))
    archives = list(Path('/kaggle/input').rglob('worcap-source.zip'))
    if extracted:
        source_root = extracted[0].parents[1]
        shutil.copytree(source_root, REPO, dirs_exist_ok=True)
    elif archives:
        REPO.mkdir(parents=True, exist_ok=True)
        shutil.unpack_archive(str(archives[0]), str(REPO))
    else:
        raise FileNotFoundError('Anexe o dataset privado worcap-source (ZIP ou extraído)')
assert (DATA / 'treino_tp.nc').exists(), 'Anexe os dados oficiais da competição'
os.environ['WORCAP_DATA_DIR'] = str(DATA)
os.environ['WORCAP_NUM_THREADS'] = '8'
os.chdir(REPO)
sys.path.insert(0, str(REPO / 'scripts'))
print('Código:', REPO)
print('Dados:', DATA)

## 0. Ambiente e testes de contrato

O teste deve passar antes de gastar GPU. O `pip freeze` e a GPU serão guardados no pacote final.

In [ ]:
ARTIFACTS = REPO / 'artifacts'
ARTIFACTS.mkdir(exist_ok=True)
subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-v'], check=True)
(ARTIFACTS / 'python-version.txt').write_text(sys.version + '\n')
with (ARTIFACTS / 'pip-freeze.txt').open('w') as handle:
    subprocess.run([sys.executable, '-m', 'pip', 'freeze'], stdout=handle, check=True)
with (ARTIFACTS / 'nvidia-smi.txt').open('w') as handle:
    subprocess.run(['nvidia-smi'], stdout=handle, check=True)
print('Ambiente registrado.')

In [ ]:
# Opcional: confirme a identidade da submissão histórica 1,69874, se ela estiver anexada.
baseline_csvs = list(Path('/kaggle/input').rglob('submission_ocean_2023_2024.csv'))
if baseline_csvs:
    subprocess.run([sys.executable, 'scripts/worcap_pipeline.py', 'audit-baseline', str(baseline_csvs[0])], check=True)
else:
    print('CSV histórico não anexado; auditoria de hash ignorada.')

## 1. Trilho LightGBM — três configurações fixas

Rode `wide`, `regularized` e `shallow` na amostra `a`. Cada comando executa os folds 1997, 2015 e 2021, sempre com janela de treino de 30 anos e bloco causal de 24 meses.

In [ ]:
subprocess.run([
    sys.executable, 'scripts/worcap_pipeline.py', '--data-dir', str(DATA),
    'validate-suite', '--configs', 'wide', 'regularized', 'shallow',
    '--folds', '1997', '2015', '2021', '--spatial-sample', 'a',
    '--n-points', '8000', '--validation-stride', '4'
], check=True)

In [ ]:
import pandas as pd
registry = pd.read_csv(REPO / 'experiments/registry.csv')
lgb = registry[registry['config'].isin(['wide', 'regularized', 'shallow'])]
display(lgb.groupby(['config', 'spatial_sample'])[['rmse_24', 'rmse_year1', 'rmse_year2']].agg(['mean', 'std']).sort_values(('rmse_year2', 'mean')))

## 2. Confirmação espacial e calibração OOF

Defina `CHOSEN_CONFIG` somente depois de revisar a tabela. Repita apenas a vencedora em `b`. A calibração mede cada fold com pesos aprendidos nos outros dois.

In [ ]:
ranking = lgb.groupby('config')[['rmse_24', 'rmse_year2']].mean().sort_values(['rmse_year2', 'rmse_24'])
CHOSEN_CONFIG = str(ranking.index[0])
print('Configuração escolhida exclusivamente pelos folds:', CHOSEN_CONFIG)
subprocess.run([
    sys.executable, 'scripts/worcap_pipeline.py', '--data-dir', str(DATA),
    'validate', '--config', CHOSEN_CONFIG, '--folds', '1997', '2015', '2021',
    '--spatial-sample', 'b', '--n-points', '8000', '--validation-stride', '4'
], check=True)

In [ ]:
for sample in ['a', 'b']:
    oof = [str(REPO / 'experiments' / f'{CHOSEN_CONFIG}_{sample}_fold{fold}_oof.npz') for fold in ['1997', '2015', '2021']]
    output = REPO / 'experiments' / f'{CHOSEN_CONFIG}_{sample}_calibration.json'
    subprocess.run([sys.executable, 'scripts/worcap_pipeline.py', 'calibrate', *oof, '--output', str(output)], check=True)
    print(sample, json.loads(output.read_text()))

In [ ]:
spatial_report = REPO / 'experiments' / f'{CHOSEN_CONFIG}_spatial_average.json'
component_a = [str(REPO / 'experiments' / f'{CHOSEN_CONFIG}_a_fold{fold}_oof.npz') for fold in ['1997', '2015', '2021']]
component_b = [str(REPO / 'experiments' / f'{CHOSEN_CONFIG}_b_fold{fold}_oof.npz') for fold in ['1997', '2015', '2021']]
subprocess.run([sys.executable, 'scripts/worcap_pipeline.py', 'evaluate-blend',
                '--component-a', *component_a, '--component-b', *component_b,
                '--spatial-average', '--output', str(spatial_report)], check=True)
SPATIAL_DECISION = json.loads(spatial_report.read_text())
display(pd.json_normalize(SPATIAL_DECISION['folds']))
print('Média a/b aprovada:', SPATIAL_DECISION['accept_average'])

## 3. U-Net experimental (gate inicial)

Execute 2015 primeiro. Não avance automaticamente: compare RMSE e blend OOF. Só então rode 2021; rode 1997 apenas se ambos passarem.

In [ ]:
subprocess.run([
    sys.executable, 'scripts/worcap_unet.py', '--data-dir', str(DATA),
    'validate', '--fold', '2015', '--max-epochs', '30', '--patience', '5'
], check=True)

## 4. Treino final LightGBM e CSV local

Só execute depois de congelar configuração, calibração e decisão sobre o ensemble `a/b`. Isto gera e audita o CSV, mas não o envia.

In [ ]:
USE_SPATIAL_ENSEMBLE = bool(SPATIAL_DECISION['accept_average'])
samples = ['a', 'b'] if USE_SPATIAL_ENSEMBLE else ['a']
metadata_paths = []
for sample in samples:
    calibration = REPO / 'experiments' / f'{CHOSEN_CONFIG}_{sample}_calibration.json'
    subprocess.run([
        sys.executable, 'scripts/worcap_pipeline.py', '--data-dir', str(DATA),
        'fit-final', '--config', CHOSEN_CONFIG, '--spatial-sample', sample,
        '--n-points', '8000', '--calibration-json', str(calibration)
    ], check=True)
    metadata_paths.append(REPO / 'artifacts' / f'lgb_{CHOSEN_CONFIG}_{sample}_final.json')
command = [sys.executable, 'scripts/worcap_pipeline.py', '--data-dir', str(DATA), 'predict']
for path in metadata_paths:
    command.extend(['--model', str(path)])
if len(metadata_paths) == 2:
    command.extend(['--weights', '0.5', '0.5'])
candidate = REPO / 'artifacts' / 'submission_candidate.csv'
command.extend(['--output', str(candidate)])
subprocess.run(command, check=True)
candidate

## 5. Manifesto final

Execute após escolher o CSV final. O manifesto liga o arquivo ao código e ao ambiente que o geraram.

In [ ]:
import hashlib
def sha256(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as handle:
        for chunk in iter(lambda: handle.read(8 << 20), b''):
            digest.update(chunk)
    return digest.hexdigest()
manifest = {}
for path in sorted(ARTIFACTS.rglob('*')):
    if path.is_file():
        manifest[str(path.relative_to(REPO))] = {'sha256': sha256(path), 'bytes': path.stat().st_size}
(ARTIFACTS / 'MANIFEST.json').write_text(json.dumps(manifest, indent=2))
print(json.dumps(manifest, indent=2))
print('Nenhum upload foi realizado. Solicite aprovação antes de submeter.')